![Cabecera](../../../Sprint_11/assets/cabecera_thebridge.png)

# Workout — Agente con tools en LangGraph (ReAct)

## Objetivo

Hasta ahora, un agente con tools suele vivir en un bucle en Python: el LLM pide una herramienta, el código la ejecuta, se vuelve a llamar al modelo, y así hasta tener una respuesta en texto.

En este notebook **dibujamos ese mismo ciclo como un grafo** LangGraph (patrón **ReAct**: Reason + Act):

```text
START → agent → ¿pide tools? → sí → tools → agent → …
                             → no → END
```

El patrón ReAct sirve para **agentes con tools**. Es utilizado en el patrón Tool Use por LangGraph.

Usaremos el estilo del ecosistema LangChain: `@tool`, `bind_tools` y `ToolNode`.

## Índice

1. Setup
2. Qué es ReAct en un grafo
3. LLM y tools a utilizar
4. Estado, nodos y router
5. Montar y compilar el grafo
6. Probar
7. Para llevarnos

**Requisito:** `GEMINI_API_KEY`.

## 1. Setup

Instalamos las librerías del grafo y del modelo Gemini vía LangChain. Descomenta la siguiente celda solo la primera vez o si falta algún paquete.

In [ ]:
# %pip install -qU langgraph langchain-google-genai langchain-core python-dotenv requests

Cargamos la API key desde un fichero `.env` (si existe) o te la pedimos de forma oculta con `getpass`.

In [1]:
import os
from getpass import getpass

from dotenv import load_dotenv

load_dotenv()

if not os.getenv("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = getpass("Pega tu GEMINI_API_KEY: ")

API_KEY = os.environ["GEMINI_API_KEY"]
print("GEMINI_API_KEY:", "sí" if API_KEY else "no")

GEMINI_API_KEY: sí


## 2. Qué es ReAct en un grafo

**ReAct** = el modelo **razona** (decide si necesita datos) y **actúa** (pide tools).  
En LangGraph eso son dos nodos y una decisión:

| Pieza | Qué hace |
|-------|----------|
| Nodo `agent` | Llama al LLM. Puede devolver texto final o `tool_calls`. |
| Nodo `tools` | Ejecuta en Python las tools pedidas. |
| Router | Tras `agent`: si hay `tool_calls` → `tools`; si no → fin. |
| Edge `tools` → `agent` | Tras ejecutar, el modelo ve los resultados y decide otra vez. |

```text
START → agent ──┬── tool_calls? sí ──► tools ──► agent
                │
                └── no ──► END
```

Idea clave: el **LLM pide**; **Python ejecuta**. El grafo solo encadena los pasos.

## 3. LLM y tools a utilizar

Definimos tres tools:

- `calcular` — expresión matemática (local).
- `obtener_clima` — llama a **Open-Meteo** (sin API key). Si falla la red → lee `data/clima_fallback.json`.
- `buscar_dato` — dato corto hardcodeado (local).

El **docstring** `@tool` de cada función importa: el modelo lo usa para saber **cuándo** llamar la tool.

`bind_tools` registra esas tools en el LLM: la respuesta puede traer `tool_calls` en lugar de solo texto.

In [12]:
import json
import math

import requests
from langchain_core.tools import tool
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=API_KEY,
    temperature=0,
)


@tool
def calcular(expresion: str) -> str:
    """Evalúa una expresión matemática sencilla.
    Ejemplo: '2 ** 10', 'math.sqrt(144)', '15 * 7 + 3'.
    """
    try:
        resultado = eval(
            expresion,
            {"__builtins__": {}},
            {"math": math, "sqrt": math.sqrt},
        )
        return f"Resultado: {resultado}"
    except Exception as e:
        return f"Error al calcular '{expresion}': {e}"


@tool
def obtener_clima(ciudad: str) -> str:
    """Clima actual de una ciudad (Open-Meteo). Úsalo si preguntan por el tiempo."""
    try:
        # 1) nombre de ciudad → latitud / longitud
        geo = requests.get(
            "https://geocoding-api.open-meteo.com/v1/search",
            params={"name": ciudad, "count": 1, "language": "es"},
            timeout=8,
        ).json()
        lugar = geo["results"][0]

        # 2) coordenadas → temperatura actual
        meteo = requests.get(
            "https://api.open-meteo.com/v1/forecast",
            params={
                "latitude": lugar["latitude"],
                "longitude": lugar["longitude"],
                "current": "temperature_2m,relative_humidity_2m",
            },
            timeout=8,
        ).json()["current"]

        return (
            f"Clima en {lugar['name']}: {meteo['temperature_2m']}°C, "
            f"humedad {meteo['relative_humidity_2m']}%"
        )
    except Exception:
        # Sin red (o ciudad no encontrada): usamos un JSON de ejemplo
        with open("data/clima_fallback.json", encoding="utf-8") as f:
            datos = json.load(f)
        c = datos.get(ciudad.lower(), datos["default"])
        return f"[sin red] Clima en {ciudad}: {c['temp_c']}°C, {c['condicion']}"


@tool
def buscar_dato(tema: str) -> str:
    """Devuelve un dato corto sobre un tema (demo local, no es una wiki real)."""
    base = {
        "madrid": "Madrid es la capital de España. Museo del Prado, Retiro.",
        "python": "Python: lenguaje creado por Guido van Rossum (1991).",
        "langgraph": "LangGraph: orquestación de agentes con grafos de estado.",
    }
    return base.get(tema.lower(), f"Dato genérico sobre '{tema}'.")


tools = [calcular, obtener_clima, buscar_dato]
llm_con_tools = llm.bind_tools(tools)
print("Tools:", [t.name for t in tools])

Tools: ['calcular', 'obtener_clima', 'buscar_dato']


## 4. Estado, nodos y router

### Estado

- `messages`: historial del ciclo (clave en inglés: así lo espera `ToolNode` por defecto). `add_messages` hace que cada nodo **añada** mensajes en lugar de pisar la lista.
- `pasos`: cuántas veces hemos pasado por el nodo `tools` (limitado por `MAX_STEPS`, que es el tope).

### Nodo agent

Construye el prompt (system + historial), llama al LLM con tools y **devuelve solo el mensaje nuevo** del modelo.

### Nodo tools

`ToolNode` lee los `tool_calls` del último mensaje y ejecuta las funciones.  
Lo envolvemos en `nodo_tools_contado` para incrementar `pasos`.

### Router

Tras `agent`:

1. Si `pasos >= MAX_STEPS` → terminar (evita bucles infinitos).
2. Si el último mensaje tiene `tool_calls` → ir a `tools`.
3. Si no → terminar (respuesta final).

In [13]:
from typing import Annotated, Literal, TypedDict

from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

MAX_STEPS = 6  # tope de vueltas agent → tools (lo decide Python, no el LLM)


class EstadoAgente(TypedDict):
    # add_messages → acumula; pasos (int) → se sustituye
    # "messages" (EN): nombre que ToolNode espera por defecto
    messages: Annotated[list, add_messages]
    pasos: int


SYSTEM = """Eres un asistente con herramientas.
Usa 'calcular' para números, 'obtener_clima' para el tiempo,
'buscar_dato' para datos factuales cortos.
Cuando tengas suficiente información, responde en texto sin más tools."""


def nodo_agente(estado: EstadoAgente) -> dict:
    """Llama al LLM; puede pedir tools o devolver texto final."""
    prompt = [SystemMessage(content=SYSTEM)] + list(estado["messages"])
    respuesta = llm_con_tools.invoke(prompt)
    if getattr(respuesta, "tool_calls", None):
        print(f"[agent] tool_calls: {[tc['name'] for tc in respuesta.tool_calls]}")
    else:
        print("[agent] respuesta final (sin tools)")
    return {"messages": [respuesta]}


nodo_tools = ToolNode(tools)


def nodo_tools_contado(estado: EstadoAgente) -> dict:
    """Ejecuta las tools pedidas y suma 1 al contador de pasos."""
    out = nodo_tools.invoke(estado)
    pasos = int(estado.get("pasos") or 0) + 1
    print(f"[tools] paso {pasos}/{MAX_STEPS}")
    return {"messages": out["messages"], "pasos": pasos}


def router_react(estado: EstadoAgente) -> Literal["tools", "__end__"]:
    """Decide el siguiente nodo tras agent."""
    if int(estado.get("pasos") or 0) >= MAX_STEPS:
        print("[router] MAX_STEPS alcanzado → END")
        return "__end__"
    ultimo = estado["messages"][-1]
    if getattr(ultimo, "tool_calls", None):
        return "tools"
    return "__end__"


print("Nodos y router listos")

Nodos y router listos


## 5. Montar y compilar el grafo

1. Registramos los nodos `agent` y `tools`.
2. Entrada: `START` → `agent`.
3. Tras `agent`, el router elige `tools` o `END`.
4. Tras `tools`, **siempre** volvemos a `agent` (cierra el ciclo ReAct).
5. `compile()` deja el grafo listo para `invoke`.

Si el entorno lo permite, imprimimos un dibujo ASCII del grafo.

In [14]:
from langgraph.graph import END, START, StateGraph

grafo = StateGraph(EstadoAgente)
grafo.add_node("agent", nodo_agente)
grafo.add_node("tools", nodo_tools_contado)

grafo.add_edge(START, "agent")
grafo.add_conditional_edges(
    "agent",
    router_react,
    {"tools": "tools", "__end__": END},
)
grafo.add_edge("tools", "agent")

app = grafo.compile()
print("Grafo compilado")
try:
    print(app.get_graph().draw_ascii())
except Exception:
    print("(ascii del grafo no disponible en este entorno)")

Grafo compilado
        +-----------+         
        | __start__ |         
        +-----------+         
               *              
               *              
               *              
          +-------+           
          | agent |           
          +-------+*          
          .         *         
        ..           **       
       .               *      
+---------+         +-------+ 
| __end__ |         | tools | 
+---------+         +-------+ 


## 6. Probar

`preguntar` arranca el grafo con un mensaje de usuario y `pasos=0`.  
Durante el ciclo verás trazas `[agent]` / `[tools]`.  
Después de cada llamada miramos lo importante del **estado** (pasos, tools pedidas, respuesta).

Primera prueba: debería usar solo `calcular`.

In [15]:
def preguntar(pregunta: str):
    print("=" * 60)
    print("PREGUNTA:", pregunta)
    return app.invoke(
        {"messages": [HumanMessage(content=pregunta)], "pasos": 0}
    )


resultado = preguntar("¿Cuánto es la raíz cuadrada de 1764?")
print("pasos tools:", resultado["pasos"])
print("nº mensajes:", len(resultado["messages"]))
print(
    "tools pedidas:",
    [tc["name"] for m in resultado["messages"] for tc in (getattr(m, "tool_calls", None) or [])],
)
print("respuesta final:", resultado["messages"][-1].content)

PREGUNTA: ¿Cuánto es la raíz cuadrada de 1764?
[agent] tool_calls: ['calcular']
[tools] paso 1/6
[agent] respuesta final (sin tools)
pasos tools: 1
nº mensajes: 4
tools pedidas: ['calcular']
respuesta final: La raíz cuadrada de 1764 es 42.


Segunda prueba: debería usar `obtener_clima`.

In [16]:
resultado = preguntar("¿Qué temperatura hace hoy en Sevilla?")
print("pasos tools:", resultado["pasos"])
print("nº mensajes:", len(resultado["messages"]))
print(
    "tools pedidas:",
    [tc["name"] for m in resultado["messages"] for tc in (getattr(m, "tool_calls", None) or [])],
)
print("respuesta final:", resultado["messages"][-1].content)

PREGUNTA: ¿Qué temperatura hace hoy en Sevilla?
[agent] tool_calls: ['obtener_clima']
[tools] paso 1/6
[agent] respuesta final (sin tools)
pasos tools: 1
nº mensajes: 4
tools pedidas: ['obtener_clima']
respuesta final: La temperatura en Sevilla es de 26.8°C con una humedad del 62%.


Tercera prueba: varias necesidades en un solo mensaje (dato + clima + cálculo).  
El grafo puede dar **varias vueltas** agent → tools → agent hasta juntar todo.

In [17]:
resultado = preguntar(
    "Busca un dato sobre Madrid y dime también su clima. "
    "Además, ¿cuánto es 3 elevado a la 8?"
)
print("pasos tools:", resultado["pasos"])
print("nº mensajes:", len(resultado["messages"]))
print(
    "tools pedidas:",
    [tc["name"] for m in resultado["messages"] for tc in (getattr(m, "tool_calls", None) or [])],
)
print("respuesta final:", resultado["messages"][-1].content)

PREGUNTA: Busca un dato sobre Madrid y dime también su clima. Además, ¿cuánto es 3 elevado a la 8?
[agent] tool_calls: ['buscar_dato']
[tools] paso 1/6
[agent] tool_calls: ['obtener_clima']
[tools] paso 2/6
[agent] tool_calls: ['calcular']
[tools] paso 3/6
[agent] respuesta final (sin tools)
pasos tools: 3
nº mensajes: 8
tools pedidas: ['buscar_dato', 'obtener_clima', 'calcular']
respuesta final: Madrid es la capital de España, conocida por el Museo del Prado y el Parque del Retiro. El clima actual en Madrid es de 21.0°C con una humedad del 52%. Además, 3 elevado a la 8 es 6561.


## 7. Para llevarnos

| Pieza | Rol |
|-------|-----|
| `@tool` + `bind_tools` | Declarar tools y enlazarlas al LLM |
| `tool_calls` | El modelo pide ejecución (no ejecuta él) |
| `ToolNode` / nodo `tools` | Python ejecuta las tools (clave de estado: `messages`) |
| Router | ¿Ir a tools o terminar? |
| `pasos` / `MAX_STEPS` | Tope del bucle en Python |
| `compile` + `invoke` | Congelar el grafo y ejecutarlo |

En este ejemplo, el nodo `tools` usa `ToolNode` (patrón oficial de LangGraph: lee `tool_calls` y ejecuta las `@tool`).

En un **proyecto** real suele ser un nodo propio que llama a tu `ejecutar_tool` + **allowlist**: el LLM pide la tool; Python decide si se ejecuta o se bloquea. Misma idea ReAct; cambia *quién* ejecuta.